# Lecture 2: Network Annotations and Identification of Key Modules

Now that we have identified the modules in our gene co-expression network, we'll proceed to the next step of the analysis: associating these modules with clinical traits. In our case, the clinical trait is whether the module is correlated with metastatic melanoma or not.

## Steps for Analysis

1. **Module Eigengene Calculation:** We'll start by calculating the module eigengene (ME) for each module. The ME is a representative gene expression profile for each module and is obtained by summarizing the first principal component of the module expression levels. This gives us a single representative profile for each module that captures the maximum variance of the gene expression data in that module.

2. **Module-Trait Relationships:** Next, we'll estimate the relationships between modules and our clinical trait of interest (melanoma vs. primary melanocytes). This is done by correlating the MEs with the clinical trait. This allows us to efficiently identify modules that have a strong relationship with the trait, i.e., modules that might be playing a significant role in melanoma.

3. **Module Significance Calculation:** To evaluate the strength of correlation, we'll calculate the module significance (MS) for each module. The MS is defined as the average absolute gene significance (GS) of all the genes in the module. The GS measures the association between gene expression and the clinical trait, and it's measured as the log10 transformation of the P value (lgP) in the linear regression between gene expression and the clinical trait.

4. **Module Annotations:** Finally, we'll identify a key module for further analysis. This module has a strong association with the clinical trait and is, therefore, of interest for further analysis. 

The above steps will help us find the key modules that are most associated with melanoma. Once these modules are identified, we can further explore the genes within these modules to uncover potential targets for therapeutic intervention or biomarkers for disease progression or prognosis.

In [ ]:
# Import required libraries, drives, etc.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import collections
import networkx as nx

from sklearn.decomposition import PCA

pd.set_option('display.precision', 2)
pd.set_option('display.max_columns',10)

# We're also going to tell Jupyter to use inline plotting instead of notebook plotting
# It basically means you don't have to use plt.show() in every cell
%matplotlib inline

# and this command will allow multiple outputs from the same cell, rather than just the last one run
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

1. **Module Eigengene Calculation:**

In the context of gene expression data analysis, a **module** refers to a group of genes that are highly interconnected based on certain criteria, such as high correlation in their expression levels. These modules often represent biological pathways or complexes, and the genes within a module often function together in a coordinated manner.

The concept of an **"eigengene"** comes from the term "eigenvalue," which is a concept in linear algebra. In the simplest terms, an eigenvalue is a scalar associated with an operation that transforms a vector in such a way that the direction of the output vector is parallel to the original vector. The associated "eigenvector" is the vector that is scaled.

When we talk about an "eigengene" of a module, we are referring to a single representative expression profile for that module, which is essentially a weighted average of the gene expression profiles of all the genes in the module. This eigengene can be thought of as a summary or a representative profile of the overall gene expression pattern within the module. The calculation of the eigengene is based on Principal Component Analysis (PCA), where the first principal component represents the eigengene.

The module eigengene has several uses. For instance, it can be correlated with external clinical traits to identify modules that are associated with these traits. In such a case, a high correlation between a module eigengene and a clinical trait would suggest that the genes in the module are coordinately up- or down-regulated in accordance with the trait.

In conclusion, the concept of the module eigengene is a powerful one because it allows us to reduce the dimensionality of our data (from many gene expression profiles down to a single profile), and it facilitates the biological interpretation of the modules by providing a single representative profile for each module.

In [ ]:

# Load the normalized expression data
expr_data = pd.read_csv('~/LECTURE_MATERIALS/DataFiles/melanoma_CountsNormal_filtered.csv', index_col=0)
# Make sure that data is in the correct format
print(expr_data.head())

from sklearn.preprocessing import StandardScaler

def calculate_module_eigengenes(expr_data, modules, base_filename):
    """Calculate the module eigengene for each module."""
    
    # Initialize a dictionary to store the module eigengenes
    module_eigengenes = {}
    
    # Initialize a scaler for standardization
    scaler = StandardScaler()
    
    # Loop through the modules
    for i, module in enumerate(modules, start=1):
        
        # Subset the expression data for the genes in the current module
        module_expr_data = expr_data[module]
        
        # Standardize the module expression data
        module_expr_data_standardized = scaler.fit_transform(module_expr_data)
        
        # Perform PCA on the standardized module expression data
        pca = PCA(n_components=1)
        module_eigengene = pca.fit_transform(module_expr_data_standardized)
        
        # Save the module eigengene to the dictionary
        module_eigengenes[f"module_{i}"] = module_eigengene.flatten()
        
        # Define the filename for the csv file
        # The filename is the base filename with the module number appended
        csv_filename = f"{base_filename}_{i}_eigengene.csv"
        
        # Save the module eigengene to a csv file
        pd.DataFrame(module_eigengene, columns=[f"module_{i}_eigengene"]).to_csv(csv_filename, index=False)
        
    # Return the module eigengenes
    return module_eigengenes


# Path to the base filename for your files
base_filename = '~/gene_module_'

# Load the modules
modules = []
for i in range(1, 14):
    module = pd.read_csv(f"{base_filename}{i}.csv", index_col=0)
    modules.append(module.index.tolist())

# Calculate the module eigengenes
module_eigengenes = calculate_module_eigengenes(expr_data, modules, base_filename)


2. **Module-Trait Relationships:**
In the context of gene expression data, a "trait" often refers to a phenotype or condition associated with the samples. In this case, the trait could be a binary condition (e.g., "primary melanocytes" vs "metastatic melanoma cells"), a categorical condition (e.g., stages of melanoma), or even a continuous variable (e.g., patient age or tumor size).
We can visualize this with a heatmap to get a qualitative assessment any module-trait relationships.

In [ ]:
# Convert the dictionary to a DataFrame
module_eigengenes_df = pd.DataFrame(module_eigengenes)

# Transpose the DataFrame so that the modules are rows and the samples are columns
module_eigengenes_df = module_eigengenes_df.T

# Assign the correct sample names to the columns of module_eigengenes_df
module_eigengenes_df.columns = expr_data.index

# Plotting
plt.figure(figsize=(10, 8))  # Set the figure size
max_abs_value = np.abs(module_eigengenes_df.values).max() # Use symmetric color limits around 0
sns.heatmap(
    module_eigengenes_df,
    cmap="coolwarm",
    center=0,
    vmin=-max_abs_value,
    vmax=max_abs_value
)  # Create a heatmap
plt.title('Module Eigengene Values Heatmap')  # Set the title of the plot
plt.xlabel('Samples')  # Set the label for the x-axis
plt.ylabel('Modules')  # Set the label for the y-axis
plt.show()  # Display the plot

The heat map has values from ~ -300 to 600. What do these values represent?

To make this Module-Trait Relationship more quantitative, we will use a statistical test that is appropriate for binary traits. The point-biserial correlation coefficient is a good choice here. The point-biserial correlation measures the strength and direction of the association that exists between one continuous variable and one binary variable.

In [ ]:
# Create a list of sample names from the expr_data DataFrame
sample_names = expr_data.index.tolist()
print(sample_names)


In [ ]:
from scipy.stats import pointbiserialr

trait = [0 if sample == 'FM_1' or sample == 'FM_2' or sample == 'FM_3' else 1 for sample in sample_names]

# Initialize dictionaries to store the module-trait correlations and p-values
module_trait_correlations = {}
module_trait_pvalues = {}

# Loop through the modules
for module, eigengene in module_eigengenes.items():
    # Calculate the point-biserial correlation between the module eigengene and the trait
    correlation, pvalue = pointbiserialr(trait, eigengene)
    # Store the correlation and p-value in the dictionaries
    module_trait_correlations[module] = correlation
    module_trait_pvalues[module] = pvalue

# Convert the dictionaries to pandas Series for easier handling
module_trait_correlations = pd.Series(module_trait_correlations)
module_trait_pvalues = pd.Series(module_trait_pvalues)

# Display the module-trait correlations and p-values
print("Module-Trait Correlations:")
print(module_trait_correlations)
print("\nModule-Trait P-values:")
print(module_trait_pvalues)


Let's visualize these results with a bar plot that shows the magnitude of the correlation and whether it is statistically significant.

In [ ]:
# Create a list of colors based on the p-values
bar_colors = ['red' if p < 0.05 else 'blue' for p in module_trait_pvalues]

# Plot the correlations with bars colored based on their statistical significance
module_trait_correlations.plot(kind='bar', color=bar_colors)
plt.title('Module-Trait Correlations')
plt.xlabel('Modules')
plt.ylabel('Correlation')

# In this plot, red bars represent modules that are significantly associated with the 
# binary trait, while blue bars represent modules that are not significantly associated.
plt.show()


What is our interpretation of this plot?

3. **Module Significance Calculation:**
Calculating the Module Significance (MS) involves the following steps:

* Define the Gene Significance (GS): This is simply the absolute value of the correlation between the gene expression profile and a clinical trait. Here, the clinical trait is binary: primary melanocytes (healthy) and metastatic melanoma. For every gene, we will calculate the correlation between its expression values across samples and the binary clinical trait, and take the absolute value.
* Calculate Module Significance (MS): This is the average GS for all genes in the module. After calculating the GS for each gene in a module, we take the average of these GS values to get the MS for the module.

In [ ]:
# Load the module 5 network data
module5_network = pd.read_csv(f"{base_filename}5.csv")

# Extract the source and target genes
source_genes = module5_network['source'].tolist()
target_genes = module5_network['target'].tolist()

# Combine the source and target genes
all_genes = source_genes + target_genes

# Create a set from the list of all genes to remove duplicates
nonredundant_genes = set(all_genes)

# Convert the set back to a list for further use
nonredundant_genes = list(nonredundant_genes)

# Print the list of non-redundant genes
print(nonredundant_genes)


In [ ]:
# Define the binary clinical trait
clinical_trait = [0 if sample == 'FM_1' or sample == 'FM_2' or sample == 'FM_3' else 1 for sample in sample_names]

# Define a dictionary to store the gene significance values
gene_significance = {}

from scipy import stats

# Calculate the gene significance for each gene
for gene in nonredundant_genes:
    gene_expr = expr_data.loc[sample_names, gene]
    correlation, _ = stats.spearmanr(gene_expr, clinical_trait)
    gene_significance[gene] = abs(correlation)

# Calculate the module significance as the average gene significance
module_significance = np.mean(list(gene_significance.values()))

print(f"Module 5 significance: {module_significance}")


##### How should we interpret these results?

The module significance is a measure of the overall correlation of the genes in a module with a specific trait or condition. In the context of your network, module significance can be interpreted as the relevance of a module to the biological condition or trait under study.

The module significance score is on a scale from -1 to 1. This suggests that the genes in module 5 are collectively associated with the binary trait you're studying (primary melanocytes vs metastatic melanoma cell lines).

Remember that this is a correlation, not a direct causation. A high module significance score means that the expression patterns of the genes in this module tend to change consistently across the two conditions, but it doesn't necessarily mean that these genes cause the differences between the conditions.

In the context of the research, this result suggests that the genes in module 5 could be important in understanding the difference between healthy cells and those from metastatic melanoma. Further biological investigation would be needed to understand the specific roles of these genes.

4. **Module Annotations:**

In our network analysis, we have identified module 3 as significantly associated with our trait of interest, which distinguishes primary melanocytes from metastatic melanoma cell lines. While we have observed a strong correlation, it's crucial to understand the biological significance behind this association. That's where gene annotation steps in. 

Gene annotation, using resources such as gene ontologies (GO), metabolic pathways, and protein-protein interactions, provides a detailed understanding of the genes' roles within a biological context. 

**Gene Ontologies (GO):** GO annotation will provide insights into the biological processes, molecular functions, and cellular components associated with the genes in module 3. This will help us understand the biological activities these genes are involved in and their location within the cell, informing us about the possible cellular processes that might be affected in metastatic melanoma.

**Metabolic Pathways:** By mapping our genes onto known metabolic pathways, we can discern if our genes of interest participate in specific biological pathways that may be relevant to melanoma metastasis. This could highlight potential metabolic changes that occur during the transition from a healthy state to a cancerous one. 

**Protein-Protein Interactions:** Our genes of interest code for proteins that don't work in isolation. They interact with other proteins to perform their functions. Understanding these interactions can elucidate complex protein networks that might be crucial in the development of metastatic melanoma. 

By annotating the genes in module 3 with these bioinformatics data, we can gain a deeper understanding of the biological context and potential functional significance of these genes in metastatic melanoma. This could provide valuable insights into the molecular mechanisms underlying melanoma metastasis and potentially identify new targets for therapeutic intervention.

There are numerous databases and websites available for gene annotation. Here are some of the most commonly used ones:

1. **NCBI Gene**: This database from the National Center for Biotechnology Information (NCBI) provides detailed information about specific genes, including gene function, gene structure, and related literature. (https://www.ncbi.nlm.nih.gov/gene)

2. **GeneCards**: This searchable database provides concise information about human genes, their products, and their involvement in diseases. It integrates gene-centric data from several hundred web sources, including databases for genomics, transcriptomics, proteomics, genetic and epigenetic data. (https://www.genecards.org/)

3. **UniProt**: The Universal Protein Resource (UniProt) is a comprehensive resource for protein sequence and annotation data. (https://www.uniprot.org/)

4. **DAVID**: The Database for Annotation, Visualization and Integrated Discovery (DAVID) provides a comprehensive set of functional annotation tools for investigators to understand the biological meaning behind a large list of genes. (https://david.ncifcrf.gov/)

5. **Ensembl**: Provides comprehensive and accurate datasets, analysis tools and visualizations for genes, transcripts, and proteins across many species. (https://www.ensembl.org/)

6. **KEGG**: The Kyoto Encyclopedia of Genes and Genomes (KEGG) is a collection of databases dealing with genomes, biological pathways, diseases, drugs, and chemical substances. (https://www.genome.jp/kegg/)

7. **Reactome**: A free, open-source, curated and peer-reviewed pathway database. The goal is to provide intuitive bioinformatics tools for the visualization, interpretation, and analysis of pathway knowledge. (https://reactome.org/)

These resources can be used to obtain functional annotation data for a given set of genes. It's important to note that not all databases will have information for all genes, and the level of detail may vary between sources.


**STRING-DB**

[STRING-DB](https://string-db.org/) is a database and web resource dedicated to protein-protein interactions, including both physical and functional interactions. It is one of the most comprehensive databases of its kind, covering approximately 25 million proteins from more than 5,000 organisms.

STRING-DB brings together information from a wide variety of sources, including experimental data, computational predictions, and public text collections, to provide a unified view of protein-protein interactions. The database assigns a confidence score to each interaction, allowing users to filter results based on the strength of the supporting evidence.

Key features of STRING-DB include:

- Coverage of a large number of species, from bacteria to humans.
- Integration of data from numerous sources, providing a comprehensive view of protein-protein interactions.
- Confidence scores for each interaction, enabling users to focus on the most reliable interactions.
- Visualization tools for exploring protein networks.
- A user-friendly interface with powerful search and filter options.
- An API that allows programmatic access to the database, facilitating its integration into bioinformatics pipelines.

For researchers studying gene networks, STRING-DB is a valuable tool for gaining insights into the functional relationships between genes. The database can help identify key players in a network, suggest potential targets for therapeutic intervention, and generate hypotheses for future experimental work.

In [ ]:
# You can interact with string-db directly with python. Here is an example of interacting with this resouce using their api.
import requests

api_url = "https://string-db.org/api"
output_format = "json"
method = "get_string_ids"
params = {
    "identifiers" : "brca1", # your protein
    "species" : 9606, # species NCBI identifier 
    "limit" : 1, # only one identifier per input protein
}

request_url = "/".join([api_url, output_format, method])

response = requests.post(request_url, data=params)

# Check if the request is successful
if response.status_code == 200:
    print(response.json())
else:
    print("Error:", response.status_code)


This output represents a successful response from the STRING database for the BRCA1 protein in Homo sapiens (human). The result is a list of dictionaries, where each dictionary contains information about one of the proteins matching the query.

Here's a breakdown of the keys and values in the dictionary:

- `'queryIndex'`: This is the index of the query item in your original request. In this case, we only made a single query, so the index is 0.

- `'queryItem'`: This is the original query item, 'brca1' in this case.

- `'stringId'`: This is the unique identifier for the protein in the STRING database. In this case, '9606.ENSP00000418960' is the STRING ID for the human BRCA1 protein.

- `'ncbiTaxonId'`: This is the taxon ID for the species in the NCBI database. In this case, 9606 corresponds to Homo sapiens (human).

- `'taxonName'`: This is the scientific name of the species. Here, it's 'Homo sapiens'.

- `'preferredName'`: This is the preferred name of the protein. In this case, 'BRCA1'.

- `'annotation'`: This is a brief description of the protein's function and role in the cell. The description for BRCA1 describes it as an E3 ubiquitin-protein ligase that plays a central role in DNA repair and is required for its tumor suppressor function.

You can use this information to annotate the genes from your module. For example, you could make additional API requests to STRING to get information about the proteins associated with each gene, or to find other proteins that interact with them.

[ShinyGO](http://bioinformatics.sdstate.edu/go/) is a web-based tool designed for Gene Ontology (GO) enrichment analysis. It's an interactive application that was developed using the Shiny framework in R, enabling users to easily perform and visualize their analyses.

ShinyGO offers the ability to perform enrichment analysis on three key categories of Gene Ontology: Biological Process, Cellular Component, and Molecular Function. Users input a list of gene names, and ShinyGO provides output on GO terms that are statistically overrepresented within the input gene set. This can provide valuable insights into the biological roles, cellular locations, and processes that the genes of interest are associated with.

In addition to basic enrichment analysis, ShinyGO offers various additional features. These include the ability to visualize enriched GO terms in a variety of formats, compare the results of multiple analyses, and customize various analysis parameters to suit the specific requirements of the user.

Refer to the latest resources and documentation available on the [ShinyGO website](http://bioinformatics.sdstate.edu/go/) for the most accurate and up-to-date information.

**Metascape**

[Metascape](http://metascape.org) is a powerful and user-friendly tool designed to provide biologists with a comprehensive gene list annotation and analysis resource. It serves as a robust platform that aids in understanding biological functions and mechanisms of gene sets derived from high-throughput experiments.

Metascape provides various features to enable deeper understanding of gene sets, such as:

- Gene Ontology (GO) enrichment
- Pathway enrichment 
- Protein-protein interaction networks 
- Membership in various biological processes and pathways

The main goal of Metascape is to facilitate efficient interpretation and annotation of large-scale datasets, thereby speeding up biological discovery.

**Reference Paper**

Zhou, Y., Zhou, B., Pache, L., Chang, M., Khodabakhshi, A. H., Tanaseichuk, O., Benner, C., & Chanda, S. K. (2019). Metascape provides a biologist-oriented resource for the analysis of systems-level datasets. Nature Communications, 10(1), 1-10. [Link to the paper](https://www.nature.com/articles/s41467-019-09234-6)

The referenced paper provides an in-depth look into the development and application of Metascape. It details how Metascape was built with the aim of providing a biologist-oriented resource for the analysis of systems-level datasets. The authors show how Metascape can be used to analyze large gene lists and how it offers a more accurate and comprehensive gene annotation and analysis resource than other available tools.

In [ ]:
# To download annotation data, let's save the list of genes in Module 5 to a csv file.
import csv
from pathlib import Path

def save_module_genes_to_csv(nonredundant_genes, filename="~/module_5_genes.csv"):
    """
    Save a list of genes to a one-column CSV file.

    Parameters
    ----------
    nonredundant_genes : list-like
        Gene names to save.
    filename : str
        Output CSV filename or path.
    """
    output_path = Path(filename).expanduser()

    with open(output_path, "w", newline="") as file:
        writer = csv.writer(file)
        for gene in nonredundant_genes:
            writer.writerow([gene])

    print(f"Saved {len(nonredundant_genes)} genes to {output_path.resolve()}")
    return output_path

save_module_genes_to_csv(nonredundant_genes)

## Tutorial: Annotating and Visualizing Network Data with Cytoscape

Cytoscape is a powerful tool that allows you to visualize complex networks, and it has the capability to incorporate additional data layers such as gene annotations. In this tutorial, we will walk you through the process of adding annotation data from the Metascape tool to a Cytoscape network.

## Step 1: Import the Network Data

1. Open Cytoscape and go to **File > Import > Network > File** from the menu.
2. Locate and select the `GeneModule_5.csv` file.
3. In the import options, make sure that source and target are mapped to source and target columns in the CSV, respectively.

## Step 2: Import Node Attributes (Annotation Data)

The Metascape results contain rich annotation data that we can add to our network as node attributes.

1. Go to **File > Import > Table > File**.
2. Select the Metascape result file `metascape_module5_results.xlsx`. Make sure you're importing the data from the 'Annotation' tab.
3. Ensure that the `name` column (or whichever column contains the gene names) is set as the key column. This is how Cytoscape will match the imported data to the nodes in the network.

## Step 3: Visualizing the Annotation Data

Now that you've imported the annotation data, you can use it to control various visual aspects of the network.
To provide more biological context to the network, we will color the nodes based on their subcellular location. This can be done by going to `Style -> Fill Color -> Column: Subcellular Location (Protein Atlas) -> Mapping Type: Discrete Mapping`. Choose the colors that best represent the different subcellular locations.

1. Go to the **Style** panel on the Control Panel (on the left side of the screen).
2. Click on `Style -> Fill Color -> Column: Subcellular Location (Protein Atlas) -> Mapping Type: Discrete Mapping`.
3. Choose the colors that best represent the different subcellular locations.

## Step 4: Layout the Network Based on Subcellular Location

Next, we will layout the network based on the subcellular location attribute. Go to the `Layout -> Group Attributes Layout -> Subcellular Location (Protein Atlas)`.

## Step 5: Analyze Network and Annotate Edges

Now, we will analyze the network to add annotation data to the edges. This can be done by going to `Tools -> NetworkAnalyzer -> Network Analysis -> Analyze Network`. It will ask if the graph is directed, select `no`.

After performing the analysis, Edge Betweenness data is added as attributes to the edges in your network.

## Step 6: Weight Edges by Edge Betweenness

Finally, let's weight the transparency of the edges by the Edge Betweenness. Go to `Style -> Edge Transparency -> Column: Edge Betweenness -> Mapping Type: Continuous Mapping`.

This tutorial provides a simple way to incorporate biological annotation data into your network and visualize it in a meaningful way. By following these steps, you will be able to gain a better understanding of the biological context of your network.


## Additional Bioinformatics Resources

Below is a list of some additional bioinformatics resources that can be helpful in various types of biological data analysis, including pathway analysis, gene set enrichment analysis, and more.

**1. Ingenuity Pathway Analysis (IPA):** IPA is a software tool that allows researchers to analyze, integrate, and understand data from gene expression, miRNA, and SNP microarrays, as well as metabolomics, proteomics, and RNAseq experiments. It can be used to model, analyze, and understand complex biological and chemical systems. More details can be found on the [IPA website](http://www.ingenuity.com/products/ipa).

**2. Gene Set Enrichment Analysis (GSEA):** GSEA is a computational method that determines whether a pre-defined set of genes shows statistically significant, concordant differences between two biological states. It's a valuable tool for interpreting the results of high-throughput experiments. There are Python libraries available such as [gseapy](https://gseapy.readthedocs.io/en/latest/) which can be used to run GSEA in Python. More details about GSEA can be found on the [GSEA website](http://www.gsea-msigdb.org/gsea/index.jsp).

**3. Bioconductor:** Bioconductor is an open-source and open-development software project for the analysis and comprehension of high-throughput genomic data. It's based on the R statistical programming language. Bioconductor includes tools for microarray and next-generation sequencing data analysis. More details can be found on the [Bioconductor website](https://www.bioconductor.org/).

**4. Biopython:** Biopython is a set of freely available tools for biological computation written in Python. It includes modules for reading and writing different sequence file formats and multiple sequence alignments, dealing with 3D macro-molecular structures, interacting with common tools such as BLAST, and more. More details can be found on the [Biopython website](https://biopython.org/).

**5. Enrichr:** Enrichr is a comprehensive gene set enrichment analysis web server updated in 2020 with 13 functional databases, a new interactive interface, new features, various machine learning models, data visualization tools, APIs, and more. More details can be found on the [Enrichr website](https://maayanlab.cloud/Enrichr/).

**6. HOMER:** HOMER (Hypergeometric Optimization of Motif EnRichment) is a suite of tools for Motif Discovery and ChIP-Seq analysis. It is a collection of command line programs for UNIX-style operating systems written in Perl and C++. More details can be found on the [HOMER website](http://homer.ucsd.edu/homer/).

**7. BRENDA:** The BRENDA Enzyme Information System represents one of the most comprehensive enzyme repositories. It is an electronic information resource comprising molecular and biochemical information on enzymes that have been classified by the IUBMB. More details can be found on the [BRENDA website](https://www.brenda-enzymes.org/).

Absolutely, here are more resources:

**8. BioGRID:** The Biological General Repository for Interaction Datasets (BioGRID) is a public database that archives and disseminates genetic and protein interaction data from model organisms and humans. More details can be found on the [BioGRID website](https://thebiogrid.org/).

**9. OmniPath:** OmniPath is a network biology resource built by integration of 102 databases. It provides various types of interactions including protein-protein binding, enzyme-PTM relationships, and transcription factor-target gene relations. More details can be found on the [OmniPath website](https://omnipathdb.org/).

**10. InWeb_InBioMap (InWeb_IM):** InWeb_InBioMap is a high-quality protein-protein interaction (PPI) network covering more than 500,000 high-confidence interactions among ~12,000 human proteins. More details can be found on the [InWeb_IM website](http://inbio-discover.com/).

**11. DisGeNET:** DisGeNET is a discovery platform containing one of the largest publicly available collections of genes and variants associated with human diseases. DisGeNET integrates data from expert curated repositories, GWAS catalogues, animal models and scientific literature. More details can be found on the [DisGeNET website](http://www.disgenet.org/).

**12. PANTHER:** The PANTHER (Protein ANalysis THrough Evolutionary Relationships) Classification System was designed to classify proteins (and their genes) in order to facilitate high-throughput analysis. More details can be found on the [PANTHER website](http://pantherdb.org/). 

**13. eggNOG:** The eggNOG database is a database of nested orthology inference across 3686 organisms. Nested orthology predictions are derived using the eggNOG algorithm, which provides orthology predictions at 3 different resolution levels (fine-grain orthology, medium-grain orthology, and broad-grain orthology). In addition, functional annotation is provided for both orthologous groups and individual genes. More details can be found on the [eggNOG website](http://eggnog5.embl.de/#/app/home). 

**14. PhosphoSitePlus® (PSP):** PhosphoSitePlus® is a comprehensive resource for investigating the structure and function of experimentally determined post-translational modifications in man and mouse. It's a knowledgebase dedicated to in-depth, manually curated information about protein phosphorylation, acetylation, and other post-translational modifications (PTMs). The information includes high-quality datasets of PTM sites, curated information about the proteins that modify and are modified, and the biological conditions under which these modifications occur. This information will be extremely valuable in understanding the functional impact of PTMs on your genes of interest. More details can be found on the [PhosphoSitePlus® website](https://www.phosphosite.org/homeAction).

Remember that the choice of resource will greatly depend on your research question and the type of analysis you want to perform. Each of these resources has its own strengths and limitations, so it's often necessary to use a combination of several tools in a single study.